### **1. Setting Environment**

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is not set in the environment variables.") 
else:
    print("GOOGLE_API_KEY is set successfully.")

GOOGLE_API_KEY is set successfully.


In [3]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
# Note : The Following Code is executed to go back to the parent directory 

%pwd            # Current working directory
%cd ..          # Change to the parent directory
%pwd            # Verify the current working directory

### **Step 2: Load PDF File**

Use `PyPDFLoader` from LangChain to load the content of the PDF file.

In [ ]:
from langchain_core.document_loaders import PyPDFLoader
filePath = "../Data/SDG.pdf"
loader = PyPDFLoader(filePath) 
data = loader.load()

In [7]:
print(data)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': '../Data/SDG.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content=''), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': '../Data/SDG.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content=''), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': '../Data/SDG.pdf', 'total_pages': 24, 'page': 2, 'page_label': '3'}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO

### **Step 3: Extract Raw Text from PDF**

Extract and concatenate the page-wise text content from the loaded document objects.

In [8]:
# Extracting text from the loaded data
# Assuming 'data' is a list of Document objects with a 'page_content' attribute

text_data = ""

for page in data:
    text_data += page.page_content
    
text_data

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

### **Step 4: Split Text into Chunks**


Use `RecursiveCharacterTextSplitter` to divide the long text into manageable chunks for further processing like embeddings or question generation.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text_data)


In [10]:
print("Type of Chunk :", type(chunks[0]))
print(f"Number of Chunks: {len(chunks)}")
print(f"First Chunk: {chunks[0]}")
print(f"\nSecond Chunk: {chunks[1]}")

Type of Chunk : <class 'str'>
Number of Chunks: 20
First Chunk: IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders from these countries created a plan called the Sustainable 
Development Goals (SDGs). This set of 17 goals imagines a future just 15 years 
off that would be rid of poverty and hunger, and safe from the worst effects of 
climate change. It’s an ambitious plan. 
But there’s ample 

### **Step 5: Create LangChain Document Objects**

Convert the text chunks into `Document` objects using LangChain's `Document` class so they can be processed for question generation or other downstream tasks.

In [ ]:
from langchain_core.documents import Document

document_question = []
for chunk in chunks:
    document_question.append(Document(page_content=chunk))
    
    
print(f"Number of Document Objects Created: {len(document_question)}")
print(f"Document Object: {document_question}")

Number of Document Objects Created: 20
Document Object: [Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. 

In [1]:
print("Type of Document Object :", type(document_question[0]))

NameError: name 'document_question' is not defined

### **Step 6: LLMChaining**

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm = GoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.1
)


prompt_template = """
You are an expert at generating questions and answers from any educational or informative content.
Your goal is to help learners understand and prepare for exams or interviews by generating high-quality questions and accurate answers from the provided content.

------------
{text}
------------

Based on the above text, generate {n} question-answer pairs that help the user prepare.
Make sure the questions cover key points and do not leave out important information.

FORMAT:
Q1: <Question 1>
A1: <Answer 1>

Q2: <Question 2>
A2: <Answer 2>

... and so on up to {n} pairs.
"""

prompt = PromptTemplate(
    input_variables=["text", "n"],
    template=prompt_template
)

chain = prompt | llm


In [17]:
solution = chain.invoke({ "text": document_question, "n": 5 })

print("Generated Questions and Answers:")
print(solution)

Generated Questions and Answers:
Q1: In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?
A1: The leaders came together in the year 2015.

Q2: What is the United Nations Development Programme (UNDP)'s role in relation to the SDGs?
A2: The UNDP is a leading organization working to fulfill the SDGs by the year 2030. It helps nations make the Goals a reality and champions the Goals so that people everywhere know how to do their part.

Q3: What is the goal related to hunger, and what does it entail?
A3: The goal is to end hunger, achieve food security and improved nutrition, and promote sustainable agriculture. This means promoting sustainable agriculture and supporting small farmers to ensure everyone has access to sufficient and nutritious food all year round.

Q4: What are some of the inequalities that women and girls still face, despite progress made?
A4: Women and girls still face gross inequalities in work and wages, lots of un

**Seperating Questions For Checking AtLast**

In [39]:
type(solution)

str

In [45]:
import re
# Extract Q-A pairs using regex
qa_pairs = re.findall(r'(Q\d+:.*?)(?:\n|$)(A\d+:.*?)(?=\nQ\d+:|\Z)', solution, re.DOTALL)

# Clean and print
for q, a in qa_pairs:
    print("Question:", q.strip())
    print("Answer:", a.strip())
    print("-" * 60)

Question: Q1: What are the Sustainable Development Goals (SDGs), and when were they created?
Answer: A1: The Sustainable Development Goals (SDGs) are a set of 17 goals created in 2015 by leaders from 193 countries. They aim to eliminate poverty and hunger and protect against the worst effects of climate change by 2030.
------------------------------------------------------------
Question: Q2: What is the role of the United Nations Development Programme (UNDP) in relation to the SDGs?
Answer: A2: The UNDP is a leading organization working to fulfill the SDGs by 2030. It operates in nearly 170 countries and territories, helping nations achieve the Goals and promoting awareness of the SDGs worldwide.
------------------------------------------------------------
Question: Q3: What are some of the specific goals outlined in the document, and what progress has been made towards achieving them?
Answer: A3: Some specific goals include ending extreme poverty, ending hunger, ensuring healthy live

### **Step 7: Generating Embedding**

In [35]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

### **Step 8: Storing in VectorDB**

In [51]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(document_question, embeddings)

In [52]:
vectorstore

### **Step 9: Retrieving**

In [ ]:
from langchain_community.chains import RetrievalQA

retriever = vectorstore.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
)

In [56]:
print(qa_chain)

verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"), llm=GoogleGenerativeAI(model='gemini-2.0-flash', google_api_key=SecretStr('**********'), temperature=0.1, client=ChatGoogleGenerativeAI(model='models/gemini-2.0-flash', google_api_key=SecretStr('**********'), temperature=0.1, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x0000021A4D39F4D0>, default_metadata=(), model_kwargs={})), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='

In [55]:
# Answer each question and save to a file
for question , a in qa_pairs:
    print("Question: ", question)
    answer = qa_chain.run(question)
    print("Model Answer: ", a)
    print("VectorDB Answer: ", answer)
    print("--------------------------------------------------\\n\\n")


Question:  Q1: What are the Sustainable Development Goals (SDGs), and when were they created?
Model Answer:  A1: The Sustainable Development Goals (SDGs) are a set of 17 goals created in 2015 by leaders from 193 countries. They aim to eliminate poverty and hunger and protect against the worst effects of climate change by 2030.

VectorDB Answer:  I don't know when the Sustainable Development Goals were created, but the context provided describes them as a "big to-do list" with 193 countries involved, aiming to achieve targets by 2030. The goals address various global challenges like poverty, hunger, health, peace, and environmental sustainability.
--------------------------------------------------\n\n
Question:  Q2: What is the role of the United Nations Development Programme (UNDP) in relation to the SDGs?
Model Answer:  A2: The UNDP is a leading organization working to fulfill the SDGs by 2030. It operates in nearly 170 countries and territories, helping nations achieve the Goals and 